In [ ]:
%pip install openpyxl pandas numpy scikit-learn matplotlib joblib

In [ ]:

# ============================================================
# MACHINE LEARNING ASSIGNMENT 2
# Dry Bean Classification
# ============================================================

# STEP 1: INSTALL REQUIRED LIBRARIES
# %pip install openpyxl pandas numpy scikit-learn matplotlib joblib


# STEP 2: IMPORT LIBRARIES

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42


# STEP 3: LOAD THE DATASET

DATA_PATH = "./dry_bean_files/DryBeanDataset/Dry_Bean_Dataset.xlsx"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}"
    )

data = pd.read_excel(DATA_PATH, engine="openpyxl")

print("Dataset loaded successfully")
print("Dataset shape:", data.shape)

display(data.head())


# STEP 4: EXPLORE THE DATASET

print("\nColumn names:")
print(data.columns.tolist())

print("\nDataset information:")
data.info()

print("\nMissing values:")
display(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())

print("\nClass distribution:")
display(data["Class"].value_counts())


# STEP 5: VISUALIZE CLASS DISTRIBUTION

plt.figure(figsize=(9, 5))

data["Class"].value_counts().plot(kind="bar")

plt.title("Distribution of Dry Bean Classes")
plt.xlabel("Bean Class")
plt.ylabel("Number of Records")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# STEP 6: SEPARATE FEATURES AND TARGET

TARGET_COLUMN = "Class"

X = data.drop(columns=[TARGET_COLUMN])
y = data[TARGET_COLUMN]

print("Number of input features:", X.shape[1])
print("Number of target classes:", y.nunique())
print("Target classes:", sorted(y.unique()))


# STEP 7: SPLIT DATA INTO TRAINING AND TESTING SETS

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nTraining data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)


# STEP 8: DEFINE THE FIVE CLASSIFICATION MODELS

models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]),

    "Gaussian Naive Bayes": GaussianNB(),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="linear",
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])
}

print("\nFive models are ready:")
for name in models:
    print(name)


# STEP 9: TRAIN AND EVALUATE ALL MODELS

results = []
trained_models = {}

os.makedirs("saved_models", exist_ok=True)

for model_name, model in models.items():

    print("\nTraining:", model_name)

    # Train model
    model.fit(X_train, y_train)

    # Predict test data
    y_pred = model.predict(X_test)

    # Calculate class probabilities for AUC
    y_score = model.predict_proba(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)

    auc = roc_auc_score(
        y_test,
        y_score,
        multi_class="ovr",
        average="macro",
        labels=model.classes_
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MCC": mcc
    })

    trained_models[model_name] = model

    # Save trained model
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_path = f"saved_models/{safe_name}.joblib"

    joblib.dump(model, model_path)

    print("Model saved:", model_path)


# STEP 10: DISPLAY MODEL COMPARISON

metrics_df = pd.DataFrame(results)

metrics_df = metrics_df.sort_values(
    by="F1",
    ascending=False
)

print("\nMODEL PERFORMANCE COMPARISON")

display(metrics_df.round(4))


# STEP 11: SAVE COMPARISON RESULTS

metrics_df.to_csv(
    "comparison_metrics.csv",
    index=False
)

print("Comparison metrics saved successfully.")


# STEP 12: SAVE TEST DATA

test_data = X_test.copy()

test_data[TARGET_COLUMN] = y_test.values

test_data.to_csv(
    "test_data.csv",
    index=False
)

print("Test data saved successfully.")


# STEP 13: IDENTIFY THE BEST MODEL

best_model_name = metrics_df.iloc[0]["Model"]

print("\nBest model based on macro F1 score:")
print(best_model_name)

best_model = trained_models[best_model_name]

best_predictions = best_model.predict(X_test)


# STEP 14: CLASSIFICATION REPORT

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        y_test,
        best_predictions,
        zero_division=0
    )
)


# STEP 15: CONFUSION MATRIX

cm = confusion_matrix(
    y_test,
    best_predictions,
    labels=best_model.classes_
)

plt.figure(figsize=(9, 7))

plt.imshow(cm, interpolation="nearest")

plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(
    range(len(best_model.classes_)),
    best_model.classes_,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(best_model.classes_)),
    best_model.classes_
)

plt.colorbar()
plt.tight_layout()
plt.show()


# STEP 16: FINAL OUTPUT FILES

print("\nASSIGNMENT MILESTONE COMPLETED")

print("\nGenerated files:")

print("1. comparison_metrics.csv")
print("2. test_data.csv")
print("3. saved_models/")

print("\nSaved models:")

for file in os.listdir("saved_models"):
    print("-", file)


In [5]:

%%writefile app.py

import os
import joblib
import streamlit as st
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# --------------------------------------------------
# 1. Streamlit page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Dry Bean Classification",
    page_icon="🌱",
    layout="wide"
)

st.title("🌱 Dry Bean Classification")
st.write(
    "This application evaluates five Machine Learning "
    "classification models using the Dry Bean Dataset."
)

# --------------------------------------------------
# 2. Model file paths
# --------------------------------------------------

MODEL_DIR = os.path.join(
    os.path.dirname(os.path.abspath(__file__)),
    "saved_models"
)

MODEL_FILES = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Gaussian Naive Bayes": "gaussian_naive_bayes.joblib",
    "SVM": "svm.joblib"
}

# --------------------------------------------------
# 3. Load saved model
# --------------------------------------------------

@st.cache_resource
def load_model(model_path):
    return joblib.load(model_path)


# --------------------------------------------------
# 4. Select model
# --------------------------------------------------

selected_model = st.selectbox(
    "Select a Machine Learning Model",
    list(MODEL_FILES.keys())
)

model_path = os.path.join(
    MODEL_DIR,
    MODEL_FILES[selected_model]
)

if not os.path.exists(model_path):
    st.error("Model file not found: " + model_path)
    st.stop()

try:
    model = load_model(model_path)
    st.success(selected_model + " model loaded successfully!")

except Exception as error:
    st.error("Error loading model: " + str(error))
    st.stop()


# --------------------------------------------------
# 5. Upload test dataset
# --------------------------------------------------

st.subheader("Upload Test Dataset")

st.write(
    "Upload a labelled CSV file containing the 16 input "
    "features and the target column named Class."
)

uploaded_file = st.file_uploader(
    "Choose a CSV file",
    type=["csv"]
)

if uploaded_file is not None:

    try:
        data = pd.read_csv(uploaded_file)

        st.write("Uploaded Dataset Preview")
        st.dataframe(data.head())

        # --------------------------------------------------
        # 6. Validate dataset
        # --------------------------------------------------

        if "Class" not in data.columns:
            st.error("The uploaded CSV must contain a 'Class' column.")
            st.stop()

        if data.empty:
            st.error("The uploaded dataset is empty.")
            st.stop()

        X = data.drop(columns=["Class"])
        y = data["Class"]

        if y.isnull().any():
            st.error("The target column contains missing values.")
            st.stop()

        if hasattr(model, "feature_names_in_"):
            expected_features = list(model.feature_names_in_)

            missing_features = [
                feature for feature in expected_features
                if feature not in X.columns
            ]

            extra_features = [
                feature for feature in X.columns
                if feature not in expected_features
            ]

            if missing_features or extra_features:
                st.error(
                    "The uploaded CSV does not have the expected "
                    "feature columns.\n\n"
                    "Missing columns: " + str(missing_features) +
                    "\n\nExtra columns: " + str(extra_features)
                )
                st.stop()

            X = X[expected_features]

        if X.isnull().any().any():
            st.error("Input features contain missing values.")
            st.stop()

        if not all(
            pd.api.types.is_numeric_dtype(X[column])
            for column in X.columns
        ):
            st.error("All input features must be numeric.")
            st.stop()

        if not np.isfinite(X.to_numpy(dtype=float)).all():
            st.error("Input features contain infinite or invalid values.")
            st.stop()

        # --------------------------------------------------
        # 7. Make predictions
        # --------------------------------------------------

        predictions = model.predict(X)

        # --------------------------------------------------
        # 8. Calculate evaluation metrics
        # --------------------------------------------------

        accuracy = accuracy_score(y, predictions)

        precision = precision_score(
            y, predictions,
            average="macro",
            zero_division=0
        )

        recall = recall_score(
            y, predictions,
            average="macro",
            zero_division=0
        )

        f1 = f1_score(
            y, predictions,
            average="macro",
            zero_division=0
        )

        mcc = matthews_corrcoef(y, predictions)

        try:
            probabilities = model.predict_proba(X)

            auc = roc_auc_score(
                y,
                probabilities,
                multi_class="ovr",
                average="macro",
                labels=model.classes_
            )

        except Exception:
            auc = None

        # --------------------------------------------------
        # 9. Display evaluation metrics
        # --------------------------------------------------

        st.subheader("Model Evaluation Results")

        col1, col2, col3 = st.columns(3)

        col1.metric("Accuracy", f"{accuracy:.4f}")
        col2.metric("Precision", f"{precision:.4f}")
        col3.metric("Recall", f"{recall:.4f}")

        col4, col5, col6 = st.columns(3)

        col4.metric("F1 Score", f"{f1:.4f}")

        if auc is not None:
            col5.metric("AUC", f"{auc:.4f}")
        else:
            col5.metric("AUC", "Not available")

        col6.metric("MCC", f"{mcc:.4f}")

        # --------------------------------------------------
        # 10. Confusion Matrix
        # --------------------------------------------------

        st.subheader("Confusion Matrix")

        labels = model.classes_

        cm = confusion_matrix(
            y,
            predictions,
            labels=labels
        )

        cm_df = pd.DataFrame(
            cm,
            index=labels,
            columns=labels
        )

        st.dataframe(cm_df)

        # --------------------------------------------------
        # 11. Classification Report
        # --------------------------------------------------

        st.subheader("Classification Report")

        report = classification_report(
            y,
            predictions,
            zero_division=0
        )

        st.text(report)

        # --------------------------------------------------
        # 12. Download predictions
        # --------------------------------------------------

        st.subheader("Download Predictions")

        results = data.copy()
        results["Predicted_Class"] = predictions

        csv_output = results.to_csv(index=False).encode("utf-8")

        st.download_button(
            label="Download Prediction Results",
            data=csv_output,
            file_name="dry_bean_predictions.csv",
            mime="text/csv"
        )

    except Exception as error:
        st.error("Something went wrong: " + str(error))

else:
    st.info("Please upload the test_data.csv file to evaluate the model.")

Overwriting app.py


In [11]:
with open("app.py", "r") as file:
    print(file.readline())

In [13]:
with open("app.py", "r") as file:
    content = file.read()

print("First 10 lines of app.py:")
print("\n".join(content.splitlines()[:10]))

First 10 lines of app.py:

import os
import joblib
import streamlit as st
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,


In [15]:
import os

project_folder = "/home/s2026ak05059/2026AK05059_MLAssignment2"

for root, dirs, files in os.walk(project_folder):
    if " .ipynb_checkpoints" in dirs:
        dirs.remove(" .ipynb_checkpoints")

    for file in files:
        print(os.path.relpath(
            os.path.join(root, file),
            project_folder
        ))

Dry_Bean_Dataset.xlsx
2026AK05059_ML_Assignment_2.ipynb
Untitled.ipynb
dry_bean_dataset.zip
comparison_metrics.csv
test_data.csv
app.py
requirements.txt
.ipynb_checkpoints/2026AK05059_ML_Assignment_2-checkpoint.ipynb
.ipynb_checkpoints/Untitled-checkpoint.ipynb
.ipynb_checkpoints/app-checkpoint.py
.ipynb_checkpoints/requirements-checkpoint.txt
dry_bean_files/DryBeanDataset/Dry_Bean_Dataset.arff
dry_bean_files/DryBeanDataset/Dry_Bean_Dataset.txt
dry_bean_files/DryBeanDataset/Dry_Bean_Dataset.xlsx
saved_models/logistic_regression.joblib
saved_models/decision_tree.joblib
saved_models/knn.joblib
saved_models/gaussian_naive_bayes.joblib
saved_models/svm.joblib
__pycache__/app.cpython-312.pyc


In [17]:
import os
import zipfile

project_folder = "/home/s2026ak05059/2026AK05059_MLAssignment2"

zip_path = "/home/s2026ak05059/2026AK05059_MLAssignment2_GitHub_Upload.zip"

files_to_include = [
    "app.py",
    "requirements.txt",
    "test_data.csv",
    "comparison_metrics.csv",
    "2026AK05059_ML_Assignment_2.ipynb"
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    # Add the main project files
    for file in files_to_include:
        full_path = os.path.join(project_folder, file)
        zipf.write(full_path, arcname=file)

    # Add all five saved models
    models_folder = os.path.join(project_folder, "saved_models")

    for file in os.listdir(models_folder):
        full_path = os.path.join(models_folder, file)

        if os.path.isfile(full_path):
            zipf.write(
                full_path,
                arcname=os.path.join("saved_models", file)
            )

print("ZIP file created successfully!")
print("Location:", zip_path)
print("ZIP size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")

ZIP file created successfully!
Location: /home/s2026ak05059/2026AK05059_MLAssignment2_GitHub_Upload.zip
ZIP size: 1.91 MB
